In [ ]:
import os
import shutil
import glob

def find_repo_root():
    direct_candidates = sorted(glob.glob('/kaggle/input/*/DiffAM'))
    if direct_candidates:
        return direct_candidates[0]

    nested_candidates = sorted(glob.glob('/kaggle/input/*/*/DiffAM'))
    if nested_candidates:
        return nested_candidates[0]

    for main_path in sorted(glob.glob('/kaggle/input/**/main.py', recursive=True)):
        candidate = os.path.dirname(main_path)
        if os.path.exists(os.path.join(candidate, 'requirements.txt')) and os.path.exists(os.path.join(candidate, 'losses', 'frequency_loss.py')):
            return candidate

    zip_candidates = sorted(glob.glob('/kaggle/input/**/*.zip', recursive=True))
    for zip_path in zip_candidates:
        extract_root = '/kaggle/working/repo_extract'
        if os.path.exists(extract_root):
            shutil.rmtree(extract_root)
        os.makedirs(extract_root, exist_ok=True)
        try:
            shutil.unpack_archive(zip_path, extract_root)
        except (shutil.ReadError, ValueError):
            continue

        extracted_diffam = sorted(glob.glob(os.path.join(extract_root, '**', 'DiffAM'), recursive=True))
        if extracted_diffam:
            return extracted_diffam[0]

        for main_path in sorted(glob.glob(os.path.join(extract_root, '**', 'main.py'), recursive=True)):
            candidate = os.path.dirname(main_path)
            if os.path.exists(os.path.join(candidate, 'requirements.txt')) and os.path.exists(os.path.join(candidate, 'losses', 'frequency_loss.py')):
                return candidate

    raise RuntimeError('Could not find the uploaded DiffAM repo under /kaggle/input. Attach your repo zip or dataset first.')


def find_celeb_root():
    patterns = [
        '/kaggle/input/*/CelebAMask-HQ',
        '/kaggle/input/*/*/CelebAMask-HQ',
        '/kaggle/input/*/CelebAMask-HQ/CelebAMask-HQ',
        '/kaggle/input/*/*/CelebAMask-HQ/CelebAMask-HQ',
    ]
    for pattern in patterns:
        matches = sorted(glob.glob(pattern))
        if matches:
            return matches[0]

    raise RuntimeError('Could not find CelebAMask-HQ under /kaggle/input. Attach the dataset in Kaggle before running this notebook.')


# 1. Copy the uploaded repo dataset into the Kaggle working directory.
repo_root = find_repo_root()
print('Using repo dataset:', repo_root)
shutil.copytree(repo_root, '/kaggle/working/DiffAM', dirs_exist_ok=True)
os.chdir('/kaggle/working/DiffAM')

# 2. Install the project and metric dependencies.
!pip install -q -r requirements.txt
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q gdown lpips scikit-image pytorch-fid
!pip uninstall -y datasets -q

# 3. Create directories used by training and evaluation.
os.makedirs('/kaggle/working/DiffAM/checkpoint', exist_ok=True)
os.makedirs('/kaggle/working/DiffAM/assets/datasets', exist_ok=True)

# 4. Download target FR models and MT assets once.
if not os.path.exists('/kaggle/working/DiffAM/assets/models'):
    print('Downloading target FR models and assets...')
    !gdown "1IKiWLv99eUbv3llpj-dOegF3O7FWW29J" -O /kaggle/working/assets.zip
    !unzip -o -q /kaggle/working/assets.zip -d /kaggle/working/assets_extracted/
    shutil.move('/kaggle/working/assets_extracted/assets', '/kaggle/working/DiffAM/assets_temp')
    shutil.copytree('/kaggle/working/DiffAM/assets_temp', '/kaggle/working/DiffAM/assets', dirs_exist_ok=True)
    !rm -rf /kaggle/working/assets.zip /kaggle/working/assets_extracted /kaggle/working/DiffAM/assets_temp

# 5. Symlink the Kaggle CelebAMask-HQ dataset.
celeb_path = find_celeb_root()
print('Using CelebAMask-HQ dataset:', celeb_path)
target_link = '/kaggle/working/DiffAM/assets/datasets/CelebAMask-HQ'
if not os.path.exists(target_link):
    os.symlink(celeb_path, target_link)

print('Environment setup and data linking complete.')


In [ ]:
import os

os.chdir('/kaggle/working/DiffAM')
os.makedirs('pretrained', exist_ok=True)

if not os.path.exists('pretrained/celeba_hq.ckpt'):
    print('Downloading base pretrained weights from the official repository...')
    !gdown --folder "1L8caY-FVzp9razKMuAt37jCcgYh3fjVU" -O /kaggle/working/DiffAM/pretrained_dl
    !mv /kaggle/working/DiffAM/pretrained_dl/*/* /kaggle/working/DiffAM/pretrained/ 2>/dev/null || mv /kaggle/working/DiffAM/pretrained_dl/* /kaggle/working/DiffAM/pretrained/
    !rm -rf /kaggle/working/DiffAM/pretrained_dl

print('Pretrained contents:')
print(os.listdir('pretrained'))


In [ ]:
import glob
import os
import shutil

os.chdir('/kaggle/working/DiffAM')
os.makedirs('runs', exist_ok=True)
os.makedirs('checkpoint', exist_ok=True)

for pattern in [
    'runs/mt_baseline_fixed*',
    'runs/mt_fft_face_union_m1*',
    'checkpoint/*mt_baseline_fixed*',
    'checkpoint/*mt_fft_face_union_m1*',
    'metrics_eval',
]:
    for path in glob.glob(pattern):
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)

for path in ['runs/baseline.log', 'runs/candidate.log']:
    if os.path.exists(path):
        os.remove(path)

print('Clean workspace ready for baseline + masked FFT comparison.')


In [ ]:
import os
import torch

os.chdir('/kaggle/working/DiffAM')
torch.cuda.empty_cache()

!PYTHONPATH=/kaggle/working/DiffAM python main.py   --makeup_transfer   --config celeba.yml   --exp /kaggle/working/DiffAM/runs/mt_baseline_fixed   --seed 1234   --do_train 1 --do_test 1   --n_train_img 200 --n_test_img 100   --n_iter 6 --t_0 60   --n_inv_step 20 --n_train_step 6 --n_test_step 6   --lr_clip_finetune 4e-6   --MT_iter_without_adv 2   --MT_adv_loss_w 1.2   --model_path pretrained/celeba_hq.ckpt   --target_img 3   --target_model 3   --ref_img 'XMY-060'   2>&1 | tee /kaggle/working/DiffAM/runs/baseline.log


In [ ]:
import os
import torch

os.chdir('/kaggle/working/DiffAM')
torch.cuda.empty_cache()

# First targeted FFT candidate from the new plan: face-union masking with stronger weights.
!PYTHONPATH=/kaggle/working/DiffAM python main.py   --makeup_transfer   --config celeba.yml   --exp /kaggle/working/DiffAM/runs/mt_fft_face_union_m1   --seed 1234   --do_train 1 --do_test 1   --n_train_img 200 --n_test_img 100   --n_iter 6 --t_0 60   --n_inv_step 20 --n_train_step 6 --n_test_step 6   --lr_clip_finetune 4e-6   --MT_iter_without_adv 2   --MT_adv_loss_w 1.2   --MT_1_fft_loss_w 3.0   --MT_2_fft_loss_w 1.5   --MT_fft_cutoff 0.10   --MT_fft_eps 1e-3   --MT_fft_mask_mode face_union   --model_path pretrained/celeba_hq.ckpt   --target_img 3   --target_model 3   --ref_img 'XMY-060'   2>&1 | tee /kaggle/working/DiffAM/runs/candidate.log


In [ ]:
import os

os.chdir('/kaggle/working/DiffAM')

!PYTHONPATH=/kaggle/working/DiffAM python scripts/compare_fid_ssim.py   --root /kaggle/working/DiffAM   --baseline-tag mt_baseline_fixed   --candidate-tag mt_fft_face_union_m1   --baseline-log /kaggle/working/DiffAM/runs/baseline.log   --candidate-log /kaggle/working/DiffAM/runs/candidate.log   --real-makeup-dir /kaggle/working/DiffAM/assets/datasets/MT-dataset/images/makeup   --device auto


In [ ]:
from PIL import Image
from IPython.display import display

fixed_grid = Image.open('/kaggle/working/DiffAM/metrics_eval/fixed_grid.png')
largest_change_grid = Image.open('/kaggle/working/DiffAM/metrics_eval/largest_change_grid.png')

print('Fixed comparison grid:')
display(fixed_grid)
print('Largest-change comparison grid:')
display(largest_change_grid)


In [ ]:
import glob
import json
import os
import re
import zipfile

os.chdir('/kaggle/working/DiffAM')

with open('metrics_eval/summary.json', 'r') as handle:
    summary = json.load(handle)

print('ASR summary:', summary['asr'])
print('SSIM summary:', summary['ssim'])
print('Pairwise FID:', summary['pairwise_fid'])
print('Standard FID:', summary['standard_fid'])


def keep_last_checkpoint(tag):
    ckpts = [p for p in glob.glob('checkpoint/*.pth') if tag in os.path.basename(p)]
    if not ckpts:
        return None

    def step_num(path):
        match = re.search(r'-(\d+)\.pth$', os.path.basename(path))
        return int(match.group(1)) if match else -1

    ckpts = sorted(ckpts, key=step_num)
    keep = ckpts[-1]
    for path in ckpts[:-1]:
        os.remove(path)
    return keep

kept = []
for tag in ['mt_baseline_fixed', 'mt_fft_face_union_m1']:
    keep = keep_last_checkpoint(tag)
    if keep:
        kept.append(keep)

for path in glob.glob('checkpoint/*.pth'):
    if path not in kept:
        os.remove(path)

print('Kept checkpoints:')
for path in kept:
    print(path)

zip_path = '/kaggle/working/diffam_ab_results.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for folder in ['checkpoint', 'runs', 'metrics_eval']:
        if not os.path.exists(folder):
            continue
        for root, _, files in os.walk(folder):
            for file_name in files:
                full_path = os.path.join(root, file_name)
                archive.write(full_path, arcname=full_path)

print('Saved:', zip_path)
